In [1]:
# conda create -n sti_stavia python=3.10 -y
# conda activate sti_stavia
!pip install pyVIA scanpy anndata pandas numpy scipy matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 10.6 MB/s eta 0:00:0000:0100:01
  Using cached pandas-2.3.3-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (12.8 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.22.4
    Uninstalling numpy-1.22.4:
      Successfully uninstalled numpy-1.22.4
  Attempting uninstall: pandas
    Found existing installation: pandas 1.4.3
    Uninstalling pandas-1.4.3:
      Successfully uninstalled pandas-1.4.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xarray-dataclass 3.0.0 requires numpy>=2.0.0, but you have numpy 1.24.4 which is incompatible.
tensorflow 2.21.0 requires numpy>=1.26.0, but you have numpy 1.24.4 which is incompatible.
stlearn 1.2.2 requires numpy<2.0,>=1.26.0, but you have numpy 1.24.4 which is incompatible.
stlearn 1.2.2 requires scanpy<2.0,>=1.11.0, but you hav

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import pyVIA.core as via

BASE = Path("../../Dataset/MERFISH/mouse_brain_map/Slice1/Replicate1/STI_results")
OUT = BASE / "StaVia"
OUT.mkdir(parents=True, exist_ok=True)

adata = sc.read_h5ad(BASE / "S1R1_prepared.h5ad")
adata.X = adata.layers["counts"].copy()

sc.pp.filter_genes(adata, min_cells=3)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(
    adata, n_top_genes=min(2000, adata.n_vars), flavor="seurat"
)
adata = adata[:, adata.var["highly_variable"]].copy()

# StaVia uses a lower-dimensional expression representation.
n_pcs = min(30, adata.n_vars - 1, adata.n_obs - 1)
sc.tl.pca(adata, n_comps=n_pcs)

# Spatially smooth expression using nearby cells.
coords = adata.obsm["spatial"]
spatial_expression = via.spatial_input(
    X_genes=adata.X,
    spatial_coords=coords,
    knn_spatial=6,
    spatial_weight=0.3,
)

# Use the spatially adjusted expression for the trajectory analysis.
from sklearn.decomposition import PCA
input_embedding = PCA(
    n_components=min(30, spatial_expression.shape[1],
                     spatial_expression.shape[0] - 1),
    random_state=42,
).fit_transform(np.asarray(spatial_expression))

# Initial clusters can help with visualization and root selection.
sc.pp.neighbors(adata, n_neighbors=15, use_rep="X_pca")
sc.tl.leiden(adata, resolution=0.5, key_added="cluster")
print(adata.obs["cluster"].value_counts())

fig, ax = plt.subplots(figsize=(8, 7))
scat = ax.scatter(
    coords[:, 0], coords[:, 1],
    c=adata.obs["cluster"].astype("category").cat.codes,
    s=3, cmap="tab20"
)
ax.invert_yaxis()
ax.set_aspect("equal")
ax.set_title("StaVia: provisional clusters")
fig.colorbar(scat, ax=ax, label="Encoded cluster")
fig.tight_layout()
fig.savefig(OUT / "01_clusters.png", dpi=200)
plt.show()

# IMPORTANT: choose a biologically plausible root cluster.
# This is a placeholder; it must be reviewed before interpretation.
ROOT_CLUSTER = "0"

if ROOT_CLUSTER not in adata.obs["cluster"].astype(str).values:
    raise ValueError("ROOT_CLUSTER is not present.")

# The cluster labels are provisional, not validated cell types.
labels = adata.obs["cluster"].astype(str).tolist()

v = via.VIA(
    data=input_embedding,
    true_label=labels,
    edgepruning_clustering_resolution=0.3,
    edgepruning_clustering_resolution_local=1,
    knn=15,
    cluster_graph_pruning=0.5,
    neighboring_terminal_states_threshold=4,
    too_big_factor=0.3,
    resolution_parameter=1,
    root_user=[ROOT_CLUSTER],
    dataset="S1R1",
    random_seed=42,
    is_coarse=True,
    preserve_disconnected=True,
    pseudotime_threshold_TS=40,
    x_lazy=0.99,
    alpha_teleport=0.99,
    edgebundle_pruning=0.5,
    edgebundle_pruning_twice=True,
    do_spatial_knn=True,
    do_spatial_layout=True,
    spatial_coords=coords,
    spatial_knn=6,
)

v.run_VIA()

# Save the fitted model and its computed results.
import pickle
with open(OUT / "stavia_model.pkl", "wb") as f:
    pickle.dump(v, f)

# Official graph visualization.
fig, ax1, ax2 = via.plot_piechart_viagraph(
    via_object=v,
    cmap_piechart="rainbow",
    cmap="plasma",
    highlight_terminal_clusters=True,
    show_legend=True,
)
fig.savefig(OUT / "02_trajectory_graph.png", dpi=200,
            bbox_inches="tight")
plt.show()

# Print likely result attributes for version-specific extraction.
print("StaVia object attributes containing 'pt', 'lineage', or 'terminal':")
for name in dir(v):
    if any(term in name.lower()
           for term in ["pt", "lineage", "terminal", "clustergraph"]):
        if not name.startswith("_"):
            print(name)

print("StaVia model saved to:", OUT)


/home/user/Gagan_env/genomicsProject/.genomicsProjectENV/lib/python3.10/site-packages/dask/dataframe/__init__.py:49: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)
I0000 00:00:1792993732.435325 3132275 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792993732.481980 3132275 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: module 'numpy' has no attribute 'dtypes'